# Coffee-Seasonality — a quantitative teardown 🔬
### Per-month HAC t-stats · Bonferroni · frost-vs-harvest spread + block-bootstrap CI · timer race · sub-period split

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Frost premium?: Busted](https://img.shields.io/badge/Frost_premium%3F-Busted-8b949e?style=flat-square)

The deep companion to [the notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We test Arabica monthly seasonality on 316 months and find no signal, a sign-flipping sub-period premium, and a timer that destroys buy-and-hold.

> ⚠️ Not investment advice. KC=F (Arabica front future, price-only, roll-naive) + 13-week T-bill (^IRX) monthly, 2000-02 → 2026-05, 316 months (Yahoo Finance, daily closes resampled to month-end, grid asserted hole-free). Offline, every cell falls back to the synthetic NULL and banners the tape. Sources in [`docs/references.md`](../docs/references.md).
>
> 💡 The `💡 In plain words` notes translate each result back into intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (coffee_seasonality/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from coffee_seasonality import data, strategy as st

# Cache-first real tape; fall back to the synthetic control OFFLINE and banner which tape we show.
try:
    from quantlab import repro
    d = repro.as_of(data.fetch_data())   # cache-first (examples/verify.py --fetch)
    if d.empty:
        raise RuntimeError("cache miss")
    TAPE = "REAL  (KC=F monthly, 2000-02..2026-05, 316 months, fp 89301021ccad)"
except Exception as e:
    d, _ = data.synthetic_world(frost_premium=0.0, seed=307)  # the NULL synthetic world
    TAPE = f"SYNTHETIC NULL control (offline fallback: {e}) -- NOT the real tape"
print("TAPE:", TAPE)
rf = d["tbill"]
timer = st.seasonal_timer(d["coffee"], tbill=rf)
net   = st.apply_costs(timer, n_trades_per_year=4, cost_bps_one_way=10)
bh    = st.buy_hold(d["coffee"])
ms    = st.month_stats(d["coffee"])
fh    = st.frost_harvest_tstat(d["coffee"])


TAPE: REAL  (KC=F monthly, 2000-02..2026-05, 316 months, fp 89301021ccad)


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | frost−harvest spread t = 0.34, 95% CI [−3.47%, +4.46%]; no month \|t\| ≥ 2 |
| Tradability | **Mirage** | timer Sharpe 0.04 vs buy-and-hold 0.20 (excess of T-bill), 0.02 net |
| Frost premium? | **Busted** | −0.49% in 2000–2012 (t = −0.64), flips to +1.35% 2013-on (t = 1.26) |

> 💡 In plain words: the most romantic seasonal story in commodities leaves no statistical fingerprint, because frost is a tail event, not a calendar event.

## 1 · The claim, steelmanned

- **H₁:** at least one frost month (Jun–Aug) has a significantly positive mean return.
- **H₂:** the frost group (Jun–Aug) has a significantly higher mean than the harvest group (May/Sep) — Welch t-test on pooled groups.
- **H₃:** the frost-minus-harvest spread's block-bootstrap 95% CI excludes zero.
- **H₄:** the long-frost / short-harvest calendar timer beats buy-and-hold (excess Sharpe).
- **H₅:** the pattern is stable across sub-periods.

## 2 · So what? — what rides on each

If H₁–H₅ hold, a fixed-month rule harvests a weather premium with no forecasting required. If they fail, the frost narrative is folklore: a few spectacular years (1975, 1994, 2021 frosts) we over-remember, with no tradable calendar.

## 3 · How we'd know — the protocol

One-sample t-stats (naive **and** Newey-West HAC) for each of the 12 calendar months vs 0 (Bonferroni threshold |t| ≈ 3 for α = 0.05/12); Welch two-sample test for frost (Jun–Aug) vs harvest (May/Sep); a circular block-bootstrap (12-month blocks, to respect the annual seasonal structure) 95% CI on the spread; the calendar timer (long frost, short harvest, T-bill otherwise — calendar-known, **no execution lag**) vs buy-and-hold, Sharpe in **excess of the T-bill on both legs**, gross and net of 10 bp/leg; and a 2000–2012 / 2013-on sub-period split.

## 4 · The teardown

### 4.1 Per-month t-stats, naive and HAC (H₁)

In [2]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print(f'TAPE: {TAPE}\n')
print(f'{"Month":6s}  {"Mean":>8s}  {"t-naive":>8s}  {"t-HAC":>8s}  {"n":>4s}  Signal?')
for m in range(1,13):
    row = ms.loc[m]
    sig = '|t|>=3 (Bonferroni)' if abs(row['tstat_hac'])>=3 else (
          '|t|>=2 (nominal)' if abs(row['tstat_hac'])>=2 else 'noise')
    print(f'{month_names[m-1]:6s}  {row["mean"]*100:+7.2f}%  {row["tstat"]:+8.2f}  '
          f'{row["tstat_hac"]:+8.2f}  {int(row["n"]):4d}  {sig}')

TAPE: REAL  (KC=F monthly, 2000-02..2026-05, 316 months, fp 89301021ccad)

Month       Mean   t-naive     t-HAC     n  Signal?
Jan       +1.68%     +1.02     +1.09    26  noise
Feb       +0.50%     +0.21     +0.30    27  noise
Mar       -0.54%     -0.31     -0.32    27  noise
Apr       +1.48%     +0.91     +1.01    27  noise
May       -1.29%     -0.62     -0.66    27  noise
Jun       -1.18%     -0.65     -0.71    26  noise
Jul       +1.22%     +0.82     +1.23    26  noise
Aug       +1.25%     +0.62     +0.64    26  noise
Sep       +1.07%     +0.64     +0.84    26  noise
Oct       -1.10%     -0.59     -0.85    26  noise
Nov       +3.84%     +1.82     +1.70    26  noise
Dec       +1.61%     +0.98     +1.00    26  noise


> 💡 In plain words: not one month clears |t| ≥ 2 on either statistic. The largest mean (November, t ≈ 1.82) isn't even part of the frost/harvest thesis, and wouldn't survive Bonferroni for 12 tests. The supposedly bullish frost month June is the most *negative* month. **H₁ rejected.**

### 4.2 Frost vs Harvest spread + block-bootstrap CI (H₂, H₃)

In [3]:
print(f'Frost (Jun-Aug):  {fh["frost_mean"]*100:+.2f}%  n={fh["n_frost"]}')
print(f'Harvest (May/Sep): {fh["harvest_mean"]*100:+.2f}%  n={fh["n_harvest"]}')
print(f'Spread: {fh["spread"]*100:+.2f}%  Welch t={fh["tstat"]:.2f}')
ci = st.spread_bootstrap_ci(d['coffee'], n_boot=5000, seed=307)
print(f'Block-bootstrap 95% CI on spread: [{ci["lo"]*100:.2f}%, {ci["hi"]*100:.2f}%]  '
      f'(point {ci["point"]*100:+.2f}%, n_boot={ci["n_boot"]})')
print('CI straddles 0:', ci['lo'] < 0 < ci['hi'])

Frost (Jun-Aug):  +0.43%  n=78
Harvest (May/Sep): -0.13%  n=53
Spread: +0.57%  Welch t=0.34


Block-bootstrap 95% CI on spread: [-3.47%, 4.46%]  (point +0.57%, n_boot=5000)
CI straddles 0: True


> 💡 In plain words: the Welch t is 0.34 and the bootstrap CI runs roughly −3.5% to +4.5% — it swamps the 0.57% point estimate. The spread is noise. **H₂ and H₃ rejected.**

### 4.3 Calendar timer vs buy-and-hold, gross and net (H₄)

In [4]:
res = {
    'frost/harvest timer (gross)': st.summary(timer, rf=rf),
    'timer (net 10bp/leg)':        st.summary(net,   rf=rf),
    'buy & hold KC=F':             st.summary(bh,    rf=rf),
}
display(pd.DataFrame(res).T[['cagr','sharpe','vol_ann','max_drawdown','n']].round(3))
print('Sharpe = excess of T-bill (^IRX), both legs, like-for-like; calendar-known rule, no lag')

,cagr,sharpe,vol_ann,max_drawdown,n
frost/harvest timer (gross),0.005,0.037,0.207,-0.751,316.0
timer (net 10bp/leg),0.001,0.017,0.207,-0.761,316.0
buy & hold KC=F,0.034,0.200,0.327,-0.693,316.0


Sharpe = excess of T-bill (^IRX), both legs, like-for-like; calendar-known rule, no lag


> 💡 In plain words: on the real tape the timer earns Sharpe ~0.04 vs ~0.20 for buy-and-hold — it destroys the (already poor) risk-adjusted return, and costs make it worse. Being systematically short into September periodically eats a frost spike. **H₄ rejected.**

### 4.4 Sub-period stability (H₅)

In [5]:
d.index = pd.DatetimeIndex(d.index)
for lab, yr in [('2000-2012',(2000,2012)), ('2013-on',(2013,2030))]:
    sl = d[(d.index.year>=yr[0]) & (d.index.year<=yr[1])]
    r = st.frost_harvest_tstat(sl['coffee'])
    print(f'{lab}: frost={r["frost_mean"]*100:+.2f}%  harvest={r["harvest_mean"]*100:+.2f}%  '
          f't={r["tstat"]:.2f}  n_fr={r["n_frost"]}  n_ha={r["n_harvest"]}')
print('\nThe frost premium FLIPS SIGN across halves -> not a stable seasonal law')

2000-2012: frost=-0.49%  harvest=+1.18%  t=-0.64  n_fr=39  n_ha=26
2013-on: frost=+1.35%  harvest=-1.40%  t=1.26  n_fr=39  n_ha=27

The frost premium FLIPS SIGN across halves -> not a stable seasonal law


> 💡 In plain words: the frost premium is *negative* in 2000–2012 (t = −0.64) and only nominally positive 2013-on (t = 1.26, still short of the bar). A premium that changes sign between halves of the sample is indistinguishable from a chance pattern. **H₅ rejected.**

## 5 · The verdict

H₁ rejected (no significant month). H₂/H₃ rejected (spread t = 0.34, CI [−3.47%, +4.46%]). H₄ rejected (timer Sharpe 0.04 vs 0.20). H₅ rejected (sign-flipping sub-periods). → Signal `NONE`, Tradability `MIRAGE`, frost premium `BUSTED`.

## 6 · Could you trade it?

No. Net of a conservative 10 bp/leg the timer is Sharpe ~0.02 and still carries the full −75% commodity drawdown. The break-even cost question is moot: there is no gross edge to defend. The whole apparatus that would harvest the frost premium instead pays you to stand in front of the occasional frost spike.

## 7 · Going further

Forks: (a) a *frost-conditioned* rule keyed to Brazilian minimum-temperature / GDD data rather than the calendar — the tail event, not the date; (b) seasonality in the KC=F **term structure** (the harvest glut may price into the curve, not spot returns); (c) cross-check Robusta (RC=F, a different geography/harvest). Companion: [226 Crude-Seasonality](../../226-crude-seasonality/).